# Phase 2: Model Training

In this notebook, I am going to build and train the Convolutional Neural Network (CNN). To save time and get higher accuracy, I will use Transfer Learning with MobileNetV2. This model is lightweight and incredibly fast, making it perfect for the real-time webcam detection I want to build in Phase 3.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os

# 1. Quick Data Reload (Since this is a new notebook kernel)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root='../data/', transform=data_transforms)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Data loaded successfully for training!")

Data loaded successfully for training!


## Building the Model (Transfer Learning)

I will load the pre-trained MobileNetV2 model. First, I'll freeze its core features so their weights don't get destroyed during my training. Then, I will replace the very last layer (the classifier) with a new one that only outputs 2 classes: Mask and No Mask.

In [2]:
# Load the pre-trained MobileNetV2
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Freeze the feature extraction layers so they aren't updated during training
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier layer
# MobileNetV2's classifier is a Sequential block, we just need to replace the last Linear layer
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 2) # 2 classes: with_mask, without_mask

# Move the model to the CPU or GPU
model = model.to(device)
print(model.classifier)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to C:\Users\UsEr/.cache\torch\hub\checkpoints\mobilenet_v2-b0353104.pth


100.0%

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)


## Defining Loss and Optimizer

To train the network, I need a way to measure how wrong its predictions are (Loss Function) and a way to update its weights to make it better (Optimizer).

In [3]:
# CrossEntropyLoss is standard for multi-class classification
criterion = nn.CrossEntropyLoss()

# Adam optimizer is efficient. I am ONLY optimizing the parameters of the new classifier layer I added.
optimizer = optim.Adam(model.classifier[1].parameters(), lr=0.001)

## The Training Loop

Here is where the actual learning happens. I'll loop through my dataset a few times (epochs). For each batch, the model makes a guess, calculates the error, and adjusts its weights. I will keep the epochs low (3) so it trains relatively quickly on my CPU.

In [4]:
import time

EPOCHS = 3

for epoch in range(EPOCHS):
    start_time = time.time()
    model.train() # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Training phase
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad() # Clear old gradients
        
        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, labels) # Calculate loss
        loss.backward() # Backpropagation
        optimizer.step() # Update weights
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100 * correct / total
    
    # Validation phase
    model.eval() # Set model to evaluation mode
    val_correct = 0
    val_total = 0
    with torch.no_grad(): # Don't calculate gradients during validation to save memory
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    end_time = time.time()
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Time: {end_time - start_time:.0f}s | "
          f"Train Loss: {running_loss/len(train_loader):.4f} | "
          f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

f:\SLIIT\_Year_03_\Semester 01\_ Self Study _\Face-Mask-Detector\.venv\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/3 | Time: 304s | Train Loss: 0.1946 | Train Acc: 92.93% | Val Acc: 98.15%
Epoch 2/3 | Time: 160s | Train Loss: 0.1000 | Train Acc: 96.61% | Val Acc: 98.74%
Epoch 3/3 | Time: 162s | Train Loss: 0.0713 | Train Acc: 97.47% | Val Acc: 98.54%


## Saving the Model

Now that the model is trained, I need to save its learned weights so I can use them later in my real-time webcam script without having to train it all over again.

In [5]:
# Ensure the saved_models directory exists
os.makedirs('../saved_models', exist_ok=True)

# Save the model's state dictionary (the learned weights)
model_path = '../saved_models/face_mask_mobilenetv2.pth'
torch.save(model.state_dict(), model_path)

print(f"Model successfully saved to {model_path}")

Model successfully saved to ../saved_models/face_mask_mobilenetv2.pth
